# Arquitectura U-Net modificada

In [1]:
import torch
import torch.nn as nn
import math

## Red neuronal

### Bloque ResNet

In [2]:
class ResnetBlock(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()
        
        if in_channels == out_channels:
            self.skip = nn.Identity()
        else:
            self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1)
            
        self.norm1 = nn.GroupNorm(num_groups=min(32, in_channels), num_channels=in_channels)
        self.act1 = nn.SiLU()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
       
        self.norm2 = nn.GroupNorm(num_groups=32, num_channels=out_channels)
        self.act2 = nn.SiLU()
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)

    def forward(self, x):
        residual = self.skip(x)
        x = self.conv1(self.act1(self.norm1(x)))
        x = self.conv2(self.act2(self.norm2(x)))
        return x + residual

In [3]:
# Ejemplo de uso:

in_channels, out_channels = 3, 64
batch_size, height, width = 8, 256, 256

conv_block = ResnetBlock(in_channels, out_channels)

x = torch.randn(batch_size, in_channels, height, width)
y = conv_block(x)

assert y.shape == (batch_size, out_channels, height, width)

### Bloque de self-attention

In [4]:
class AttentionBlock(nn.Module):
    
    def __init__(self, channels):
        super().__init__()
        self.norm = nn.GroupNorm(num_groups=32, num_channels=channels)
        self.q = nn.Conv2d(channels, channels, kernel_size=1)
        self.k = nn.Conv2d(channels, channels, kernel_size=1)
        self.v = nn.Conv2d(channels, channels, kernel_size=1)
        self.proj = nn.Conv2d(channels, channels, kernel_size=1)

    def forward(self, x):
        B, C, H, W = x.shape
        x_norm = self.norm(x)

        Q = self.q(x_norm).flatten(start_dim=2).transpose(1, 2)  # [B, HW, C].
        K = self.k(x_norm).flatten(start_dim=2).transpose(1, 2)  # [B, HW, C].
        V = self.v(x_norm).flatten(start_dim=2).transpose(1, 2)  # [B, HW, C].

        scores = Q @ K.transpose(1, 2) / math.sqrt(C)
        attn = scores.softmax(dim=-1)
        out = attn @ V
        out = out.transpose(1, 2).view(B, C, H, W)

        return x + self.proj(out)

In [5]:
# Ejemplo de uso:

channels = 64
batch_size, height, width = 8, 16, 16

attention_block = AttentionBlock(channels)

x = torch.randn(batch_size, channels, height, width)
y = attention_block(x)

assert y.shape == (batch_size, channels, height, width)

### Bloque downsampling

In [6]:
class DownBlock(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = ResnetBlock(in_channels, out_channels)
        self.down = nn.MaxPool2d(kernel_size=2)

    def forward(self, x):
        skip = self.conv(x)
        output = self.down(skip)
        return output, skip

In [7]:
# Ejemplo de uso:

in_channels, out_channels = 32, 64
batch_size, height, width = 8, 256, 256

down_block = DownBlock(in_channels, out_channels)

x = torch.randn(batch_size, in_channels, height, width)
y, skip = down_block(x)

assert y.shape == (batch_size, out_channels, height // 2, width // 2)
assert skip.shape == (batch_size, out_channels, height, width)

### Bloque upsampling

In [8]:
class UpBlock(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, in_channels//2, kernel_size=2, stride=2)
        self.conv = ResnetBlock(in_channels, out_channels)

    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

In [9]:
# Ejemplo de uso:

in_channels, out_channels = 64, 32
batch_size, height, width = 8, 256, 256

up_block = UpBlock(in_channels, out_channels)

x = torch.randn(batch_size, in_channels, height, width)
skip = torch.randn(batch_size, in_channels // 2, 2*height, 2*width)
y = up_block(x, skip)

assert y.shape == (batch_size, out_channels, 2*height, 2*width)

### Arquitectura U-Net modificada

In [10]:
class UNetResAtt(nn.Module):

    def __init__(self, in_channels, n_classes, base_ch=64):
        super().__init__()
        
        self.down1 = DownBlock(in_channels, base_ch)
        self.down2 = DownBlock(base_ch, base_ch*2)
        self.down3 = DownBlock(base_ch*2, base_ch*4)
        self.down4 = DownBlock(base_ch*4, base_ch*8)
        
        self.bottleneck = nn.Sequential(
            ResnetBlock(base_ch*8, base_ch*16),
            AttentionBlock(base_ch*16)
        )

        self.up4 = UpBlock(base_ch*16, base_ch*8)
        self.up3 = UpBlock(base_ch*8, base_ch*4)
        self.up2 = UpBlock(base_ch*4, base_ch*2)
        self.up1 = UpBlock(base_ch*2, base_ch)

        self.out = nn.Conv2d(base_ch, n_classes, kernel_size=1)

    def forward(self, x):
        x, skip1 = self.down1(x)
        x, skip2 = self.down2(x)
        x, skip3 = self.down3(x)
        x, skip4 = self.down4(x)

        x = self.bottleneck(x)

        x = self.up4(x, skip4)
        x = self.up3(x, skip3)
        x = self.up2(x, skip2)
        x = self.up1(x, skip1)

        return self.out(x)

In [11]:
# Ejemplo de uso:

in_channels, n_classes = 3, 1
batch_size, height, width = 8, 256, 256

unet = UNetResAtt(in_channels, n_classes)

x = torch.randn(batch_size, in_channels, height, width)
y = unet(x)
assert y.shape == (batch_size, n_classes, height, width)